# 🛣️ JalanCerdas AI — Full Training Pipeline

**⚠️ PENTING:** Jalankan dengan GPU!
```
Runtime → Change runtime type → T4 GPU
```

## 1️⃣ Install & Setup

In [ ]:
!pip install -q ultralytics kagglehub pyyaml tqdm

import torch, os
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("NO GPU! Set Runtime > Change runtime type > GPU")

## 2️⃣ Download Dataset

In [ ]:
import kagglehub, os

print("Downloading dataset...")
path = kagglehub.dataset_download("habibiahmadim09/kerusakan-jalan")
DATASET_SRC = os.path.join(path, "kerusakan-jalan")

for split in ["train", "valid", "test"]:
    sp = os.path.join(DATASET_SRC, split)
    if os.path.exists(sp):
        imgs = len([f for f in os.listdir(sp) if f.endswith((".jpg",".jpeg",".png"))])
        print(f"  {split}: {imgs} images")

print(f"\nSource: {DATASET_SRC}")

## 3️⃣ Convert VOC XML → YOLO + Download

In [ ]:
import os, shutil, xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict
from google.colab import files

CLASS_NAMES = ["retak_memanjang","pengelupasan_lapisan_permukaan","lubang","retak_kulit_buaya","retak_blok","retak_pinggir"]
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

def voc_to_yolo(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    w = int(root.find("size").find("width").text)
    h = int(root.find("size").find("height").text)
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in CLASS_TO_ID: continue
        bb = obj.find("bndbox")
        xmin, ymin = float(bb.find("xmin").text), float(bb.find("ymin").text)
        xmax, ymax = float(bb.find("xmax").text), float(bb.find("ymax").text)
        xc = max(0, min(1, ((xmin+xmax)/2)/w))
        yc = max(0, min(1, ((ymin+ymax)/2)/h))
        bw = max(0, min(1, (xmax-xmin)/w))
        bh = max(0, min(1, (ymax-ymin)/h))
        lines.append(f"{CLASS_TO_ID[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines

DATASET = "/content/dataset"
counts = defaultdict(int)
total_img, total_lbl = 0, 0

for split in ["train", "valid", "test"]:
    src = Path(DATASET_SRC) / split
    if not src.exists(): continue
    dst_name = "val" if split == "valid" else split
    dst_img = Path(DATASET) / dst_name / "images"
    dst_lbl = Path(DATASET) / dst_name / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    imgs = [f for f in src.iterdir() if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}]
    print(f"Processing {split}: {len(imgs)} images...")
    for img in imgs:
        shutil.copy2(img, dst_img / img.name)
        total_img += 1
        xml = img.with_suffix(".xml")
        if xml.exists():
            yolo = voc_to_yolo(xml)
            if yolo:
                (dst_lbl / (img.stem + ".txt")).write_text("\n".join(yolo))
                total_lbl += len(yolo)
                for l in yolo:
                    counts[CLASS_NAMES[int(l.split()[0])]] += 1
    print(f"  Done")

print(f"\nTotal: {total_img} images, {total_lbl} labels")
for n, c in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {n}: {c}")

# Download dataset ZIP
print("\nZipping dataset...")
shutil.make_archive("dataset_yolo", 'zip', '.', DATASET)
zip_size = os.path.getsize("dataset_yolo.zip") / (1024*1024)
print(f"dataset_yolo.zip ({zip_size:.1f} MB)")
files.download("dataset_yolo.zip")
print("Downloaded!")

## 4️⃣ Create data.yaml

In [ ]:
import yaml

data_yaml = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 6,
    'names': {
        0: 'retak_memanjang',
        1: 'pengelupasan_lapisan_permukaan',
        2: 'lubang',
        3: 'retak_kulit_buaya',
        4: 'retak_blok',
        5: 'retak_pinggir'
    }
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("data.yaml created!")
print(open('/content/data.yaml').read())

## 5️⃣ Train YOLOv8n (100 Epochs) + Download

In [ ]:
from ultralytics import YOLO
import torch, os, shutil
from google.colab import files

device = "0" if torch.cuda.is_available() else "cpu"
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("\nLoading YOLOv8n...")
model = YOLO("yolov8n.pt")

print("\nStarting training (100 epochs)...\n")
results = model.train(
    data="/content/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    project="/content/runs",
    name="train",
    exist_ok=True,
    patience=20,
    save=True,
    save_period=10,
    workers=4,
)

# Find weights
WEIGHTS_DIR = None
for root, dirs, files_list in os.walk("/content/runs"):
    if "best.pt" in files_list:
        WEIGHTS_DIR = root
        break

BEST_PT = os.path.join(WEIGHTS_DIR, "best.pt") if WEIGHTS_DIR else None
LAST_PT = os.path.join(WEIGHTS_DIR, "last.pt") if WEIGHTS_DIR else None

print(f"\nWeights dir: {WEIGHTS_DIR}")
print(f"best.pt: {os.path.exists(BEST_PT) if BEST_PT else False}")

# Download model
print("\nDownloading trained model...")
os.makedirs("trained_model", exist_ok=True)
if BEST_PT and os.path.exists(BEST_PT):
    shutil.copy2(BEST_PT, "trained_model/best.pt")
    print(f"  best.pt ({os.path.getsize(BEST_PT)/1024/1024:.1f} MB)")
if LAST_PT and os.path.exists(LAST_PT):
    shutil.copy2(LAST_PT, "trained_model/last.pt")
    print(f"  last.pt ({os.path.getsize(LAST_PT)/1024/1024:.1f} MB)")

results_csv = os.path.join(WEIGHTS_DIR, "..", "results.csv") if WEIGHTS_DIR else None
if results_csv and os.path.exists(results_csv):
    shutil.copy2(results_csv, "trained_model/results.csv")
    print("  results.csv")

shutil.make_archive("trained_model", 'zip', '.', "trained_model")
print(f"\ntrained_model.zip ({os.path.getsize('trained_model.zip')/1024/1024:.1f} MB)")
files.download("trained_model.zip")
print("\nModel downloaded!")

## 6️⃣ Evaluate + Download

In [ ]:
from ultralytics import YOLO
import torch, os, shutil
from google.colab import files

model = YOLO(BEST_PT)
print("Evaluating...")
metrics = model.val(data="/content/data.yaml", imgsz=640, batch=16, device="0" if torch.cuda.is_available() else "cpu")

print(f"\nmAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

names = ["retak_memanjang","pengelupasan_lapisan_permukaan","lubang","retak_kulit_buaya","retak_blok","retak_pinggir"]
print("\nPer-class:")
for i, (n, ap) in enumerate(zip(names, metrics.box.maps)):
    print(f"  {i}. {n}: {ap:.4f}")

# Download evaluation
os.makedirs("evaluation", exist_ok=True)
eval_dir = os.path.join(WEIGHTS_DIR, "..") if WEIGHTS_DIR else None
if eval_dir:
    for f in ["results.csv", "confusion_matrix.png", "results.png", "F1_curve.png", "PR_curve.png"]:
        src = os.path.join(eval_dir, f)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join("evaluation", f))
            print(f"  {f}")

shutil.make_archive("evaluation", 'zip', '.', "evaluation")
print(f"\nevaluation.zip ({os.path.getsize('evaluation.zip')/1024/1024:.1f} MB)")
files.download("evaluation.zip")
print("Evaluation downloaded!")

## 7️⃣ Export TFLite + Download

In [ ]:
from ultralytics import YOLO
import os, shutil
from google.colab import files

model = YOLO(BEST_PT)
print("Exporting TFLite...")
model.export(format="tflite", imgsz=640, half=True, nms=True)

# Find TFLite
tflite_path = None
for root, dirs, fnames in os.walk(WEIGHTS_DIR):
    for f in fnames:
        if f.endswith(".tflite"):
            tflite_path = os.path.join(root, f)
            break
    if tflite_path: break

# Download
os.makedirs("mobile_model", exist_ok=True)
if tflite_path and os.path.exists(tflite_path):
    shutil.copy2(tflite_path, "mobile_model/best.tflite")
    print(f"best.tflite ({os.path.getsize(tflite_path)/1024/1024:.1f} MB)")

for root, dirs, fnames in os.walk(WEIGHTS_DIR):
    for f in fnames:
        if f.endswith(".onnx"):
            shutil.copy2(os.path.join(root, f), "mobile_model/best.onnx")
            print(f"best.onnx")
            break

if os.path.exists(BEST_PT):
    shutil.copy2(BEST_PT, "mobile_model/best.pt")
    print(f"best.pt ({os.path.getsize(BEST_PT)/1024/1024:.1f} MB)")

shutil.make_archive("mobile_model", 'zip', '.', "mobile_model")
print(f"\nmobile_model.zip ({os.path.getsize('mobile_model.zip')/1024/1024:.1f} MB)")
files.download("mobile_model.zip")
print("Mobile model downloaded!")

## 8️⃣ Test Prediction + Download

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os, shutil
from google.colab import files

model = YOLO(BEST_PT)
val_dir = "/content/dataset/val/images"
imgs = [f for f in os.listdir(val_dir) if f.endswith((".jpg",".png"))][:5]

print("Testing predictions...")
for img in imgs:
    results = model.predict(source=os.path.join(val_dir, img), imgsz=640, conf=0.5, save=True, project="/content/runs", name="predict", exist_ok=True)
    print(f"\n{img}:")
    for r in results:
        for box in r.boxes:
            print(f"  {model.names[int(box.cls[0])]}: {float(box.conf[0]):.0%}")

pred_dir = "/content/runs/predict"
if os.path.exists(pred_dir):
    pred_files = [f for f in os.listdir(pred_dir) if f.endswith((".jpg",".png"))]
    if pred_files:
        display(Image(os.path.join(pred_dir, pred_files[-1]), width=600))

# Download predictions
if os.path.exists(pred_dir):
    shutil.make_archive("predictions", 'zip', '.', pred_dir)
    print(f"\npredictions.zip ({os.path.getsize('predictions.zip')/1024/1024:.1f} MB)")
    files.download("predictions.zip")
    print("Predictions downloaded!")